In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# Финальное обучение на train + val

Пользователь подтвердил разрешение train+val. Объединяем 4999+699=5698 размеченных изображений, не перемещая и не меняя raw. Это **финальный** этап после фиксации pipeline. Рецепт partial+stronger сохраняется; число эпох заранее фиксировано по исходному лучшему checkpoint (50:20, 101:21), исходный cosine T_max=25 сохранён.

Инициализация из ImageNet V2, не из обученного на этих данных best.pt: это refit выбранного рецепта на объединённой выборке. Не запускается автоматически. Нужен Kaggle GPU; это единственная тяжёлая часть плана. Никакой Accuracy на бывшей val, early stopping или best.pt; сохраняется **final.pt последней заданной эпохи**. Test только для опциональных predictions.

Два presets нужны для воспроизведения двух участников прежнего ансамбля; не обязательно обучать оба, если выбран одиночный pipeline. Новые reranker/ensemble варианты не следует автоматически добавлять в refit до анализа их результатов.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from archive.src.final_refit import run_final_refit
from archive.src.experiment import load_config

CONFIG = 'configs/resnet101_final_train_val.json'  # либо resnet50_final_train_val.json
RUN_REFIT = False  # включить только для финального запуска на Kaggle
configuration = load_config(CONFIG, PROJECT_ROOT)
display(configuration)


In [ ]:
if RUN_REFIT:
    result = run_final_refit(configuration, project_root=PROJECT_ROOT,
                             predict_test=False, show_plots=True)
    display(result['summary'])
else:
    print('Final refit disabled. No training or test inference performed.')


Артефакты: artifacts/models/<id>/final.pt; logs/config.json, metadata.json, history.csv, summary.json, training.log, learning_curves.png. В experiments.csv добавляется строка с **пустой val_accuracy** и пометкой FINAL train+val. Если PREDICT_TEST=True, сохраняются test probabilities и submission.

Старые честные train/val результаты остаются в журнале отдельно. По ним уже выбран рецепт; новые refit веса нельзя оценивать на бывшей val как на независимой выборке.
